In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/sep-25-dl-gen-ai-nppe-2/sample_submission.csv
/kaggle/input/sep-25-dl-gen-ai-nppe-2/train.csv
/kaggle/input/sep-25-dl-gen-ai-nppe-2/test.csv
/kaggle/input/birnn-dataset-v1/v1
/kaggle/input/bigru-dataset-v1/v1).


# Initial Setup and File Exploration


In [2]:
# Cell 1: Initial Setup and File Exploration
import numpy as np
import pandas as pd
import os

# List input files
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/sep-25-dl-gen-ai-nppe-2/sample_submission.csv
/kaggle/input/sep-25-dl-gen-ai-nppe-2/train.csv
/kaggle/input/sep-25-dl-gen-ai-nppe-2/test.csv
/kaggle/input/birnn-dataset-v1/v1
/kaggle/input/bigru-dataset-v1/v1).


# Install Required Packages


In [3]:
# Cell 2: Install Required Packages
!pip install trackio -qq
!pip install kagglehub -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 988.0/988.0 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 104.6 MB/s eta 0:00:00


# Library Imports

In [4]:
# Cell 3: Import All Required Libraries
import numpy as np
import pandas as pd
import torch 
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from pytorch_lightning import Trainer, seed_everything
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
import pytorch_lightning as pl
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import kagglehub
import trackio
import warnings
import os
warnings.filterwarnings('ignore')

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("dra_hf_access_token")


# Set Data Path and Random State


In [5]:
# Cell 4: Set Data Path and Random State
DATA_PATH = "/kaggle/input/sep-25-dl-gen-ai-nppe-2"
RANDOM_STATE = 42

seed_everything(RANDOM_STATE)

Seed set to 42


42

# 2. Data Loading & Inspection


In [6]:
# Cell 5: Load Data
train_df = pd.read_csv(f"{DATA_PATH}/train.csv")
test_df = pd.read_csv(f"{DATA_PATH}/test.csv")

print(f"Train dataset shape: {train_df.shape}")
print(f"Test dataset shape: {test_df.shape}")
print("\nTrain columns:", train_df.columns.tolist())
print("\nFirst few rows:")
print(train_df.head())

# Check sequence lengths
train_df['seq_length'] = train_df['seq'].apply(len)
print(f"\nSequence length statistics:")
print(train_df['seq_length'].describe())

Train dataset shape: (7262, 4)
Test dataset shape: (1816, 2)

Train columns: ['id', 'seq', 'sst8', 'sst3']

First few rows:
   id                                                seq  \
0   0                        GVGLEGGVQLSPARTRGPEFAAPEQAG   
1   1  NHGKVKIEHTKWNVEYKVTYNRNVFANHIRSGELASNGYHTTRRTA...   
2   2  EMRKMLADWKGLSKSDGMLSSEGRTKALWLGEANFSYVPKLDPRAS...   
3   3  QDNKNGWQIRSDDVWGPDTKDSIQTVEGTRDNVVVYKGPSGYVTAP...   
4   4  VPNSRDGGGGNHWNVEFGQLIALIGAAICGVIGGALGGFTAAGSCG...   

                                                sst8  \
0                        CCCCCCCCSCCCCCCGGGCCCCCCCCC   
1  CEEEEEECCTTTEEEEEEEEEEEEEEEEEEEEECSCCSSSCCCCEE...   
2  CTHHHHHHHHHSGGGCCCCCCCCCCCCEEECSSCEEEEEEETTCGG...   
3  CCCCCCEECTTCCBCTTCTTCCEEEEBCTTTEEEEEETTEEEEEEE...   
4  CEEEEEECCCHHHHHHTHHHHHHHHHHHHHHHHSCCTTCCCCSSCC...   

                                                sst3  
0                        CCCCCCCCCCCCCCCHHHCCCCCCCCC  
1  CEEEEEECCCCCEEEEEEEEEEEEEEEEEEEEECCCCCCCCCCCEE...  
2  CCHHHHHHHH

# Create Vocabularies


In [7]:
# Cell 6: Create Vocabularies
# Amino acid vocabulary (20 standard + masked)
amino_acids = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 
               'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y', '*']

# Q8 labels
q8_labels = ['H', 'G', 'I', 'E', 'B', 'T', 'S', 'C']

# Q3 labels
q3_labels = ['H', 'E', 'C']

# Create mappings
aa_to_idx = {aa: idx for idx, aa in enumerate(amino_acids)}
idx_to_aa = {idx: aa for aa, idx in aa_to_idx.items()}

q8_to_idx = {label: idx for idx, label in enumerate(q8_labels)}
idx_to_q8 = {idx: label for label, idx in q8_to_idx.items()}

q3_to_idx = {label: idx for idx, label in enumerate(q3_labels)}
idx_to_q3 = {idx: label for label, idx in q3_to_idx.items()}

# Q8 to Q3 mapping
q8_to_q3_map = {
    'H': 'H', 'G': 'H', 'I': 'H',  # Helix
    'E': 'E', 'B': 'E',              # Strand
    'C': 'C', 'S': 'C', 'T': 'C'     # Coil
}

print("Vocabulary sizes:")
print(f"Amino acids: {len(amino_acids)}")
print(f"Q8 labels: {len(q8_labels)}")
print(f"Q3 labels: {len(q3_labels)}")

Vocabulary sizes:
Amino acids: 21
Q8 labels: 8
Q3 labels: 3


# Dataset Class


In [8]:
# Cell 7: Dataset Class
class ProteinDataset(Dataset):
    def __init__(self, df, aa_to_idx, q8_to_idx, q3_to_idx, train=True):
        self.df = df.reset_index(drop=True)
        self.aa_to_idx = aa_to_idx
        self.q8_to_idx = q8_to_idx
        self.q3_to_idx = q3_to_idx
        self.train = train
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Encode sequence
        seq = row['seq']
        seq_encoded = torch.tensor([self.aa_to_idx.get(aa, self.aa_to_idx['*']) 
                                     for aa in seq], dtype=torch.long)
        
        if self.train:
            # Encode Q8 labels
            sst8 = row['sst8']
            sst8_encoded = torch.tensor([self.q8_to_idx[label] for label in sst8], 
                                        dtype=torch.long)
            
            # Encode Q3 labels
            sst3 = row['sst3']
            sst3_encoded = torch.tensor([self.q3_to_idx[label] for label in sst3], 
                                        dtype=torch.long)
            
            return seq_encoded, sst8_encoded, sst3_encoded, len(seq)
        else:
            return seq_encoded, row['id'], len(seq)

def collate_fn(batch):
    if len(batch[0]) == 4:  # Training data
        seqs, sst8s, sst3s, lengths = zip(*batch)
        lengths = torch.tensor(lengths)
        
        # Pad sequences
        seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=0)
        sst8_padded = pad_sequence(sst8s, batch_first=True, padding_value=-1)
        sst3_padded = pad_sequence(sst3s, batch_first=True, padding_value=-1)
        
        return seqs_padded, sst8_padded, sst3_padded, lengths
    else:  # Test data
        seqs, ids, lengths = zip(*batch)
        lengths = torch.tensor(lengths)
        seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=0)
        return seqs_padded, ids, lengths

print("✓ Dataset class created")

✓ Dataset class created


# Split Data and Create DataLoaders


In [9]:
# Cell 8: Split Data and Create DataLoaders
train_df_split, val_df = train_test_split(train_df, test_size=0.1, random_state=RANDOM_STATE)

train_dataset = ProteinDataset(train_df_split, aa_to_idx, q8_to_idx, q3_to_idx, train=True)
val_dataset = ProteinDataset(val_df, aa_to_idx, q8_to_idx, q3_to_idx, train=True)
test_dataset = ProteinDataset(test_df, aa_to_idx, q8_to_idx, q3_to_idx, train=False)

BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                         collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                       collate_fn=collate_fn, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                        collate_fn=collate_fn, num_workers=2)

print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")

Train size: 6535
Val size: 727
Test size: 1816


# Bidirectional RNN Model


In [10]:
# Cell 9: Bidirectional RNN Model
class BiRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, 
                 num_q8_classes, num_q3_classes, num_layers=2, dropout=0.3):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        self.rnn = nn.RNN(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.dropout = nn.Dropout(dropout)
        self.fc_q8 = nn.Linear(hidden_dim * 2, num_q8_classes)
        self.fc_q3 = nn.Linear(hidden_dim * 2, num_q3_classes)
        
    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = pack_padded_sequence(embedded, lengths.cpu(), 
                                     batch_first=True, enforce_sorted=False)
        packed_output, _ = self.rnn(packed)
        output, _ = pad_packed_sequence(packed_output, batch_first=True)
        output = self.dropout(output)
        
        q8_logits = self.fc_q8(output)
        q3_logits = self.fc_q3(output)
        
        return q8_logits, q3_logits

print("✓ BiRNN model defined")

✓ BiRNN model defined


# Bidirectional LSTM Model


In [11]:
# Cell 10: Bidirectional LSTM Model
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, 
                 num_q8_classes, num_q3_classes, num_layers=2, dropout=0.3):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.dropout = nn.Dropout(dropout)
        self.fc_q8 = nn.Linear(hidden_dim * 2, num_q8_classes)
        self.fc_q3 = nn.Linear(hidden_dim * 2, num_q3_classes)
        
    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = pack_padded_sequence(embedded, lengths.cpu(), 
                                     batch_first=True, enforce_sorted=False)
        packed_output, _ = self.lstm(packed)
        output, _ = pad_packed_sequence(packed_output, batch_first=True)
        output = self.dropout(output)
        
        q8_logits = self.fc_q8(output)
        q3_logits = self.fc_q3(output)
        
        return q8_logits, q3_logits

print("✓ BiLSTM model defined")

✓ BiLSTM model defined


# Bidirectional GRU Model


In [12]:
# Cell 11: Bidirectional GRU Model
class BiGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, 
                 num_q8_classes, num_q3_classes, num_layers=2, dropout=0.3):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.dropout = nn.Dropout(dropout)
        self.fc_q8 = nn.Linear(hidden_dim * 2, num_q8_classes)
        self.fc_q3 = nn.Linear(hidden_dim * 2, num_q3_classes)
        
    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = pack_padded_sequence(embedded, lengths.cpu(), 
                                     batch_first=True, enforce_sorted=False)
        packed_output, _ = self.gru(packed)
        output, _ = pad_packed_sequence(packed_output, batch_first=True)
        output = self.dropout(output)
        
        q8_logits = self.fc_q8(output)
        q3_logits = self.fc_q3(output)
        
        return q8_logits, q3_logits

print("✓ BiGRU model defined")

✓ BiGRU model defined


# PyTorch Lightning Module


In [13]:
# Cell 12: PyTorch Lightning Module
class ProteinStructureModel(pl.LightningModule):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.criterion = nn.CrossEntropyLoss(ignore_index=-1)
        
        self.train_q8_preds = []
        self.train_q8_targets = []
        self.train_q3_preds = []
        self.train_q3_targets = []
        
        self.val_q8_preds = []
        self.val_q8_targets = []
        self.val_q3_preds = []
        self.val_q3_targets = []
        
    def forward(self, x, lengths):
        return self.model(x, lengths)
    
    def training_step(self, batch, batch_idx):
        seqs, sst8, sst3, lengths = batch
        q8_logits, q3_logits = self(seqs, lengths)
        
        q8_logits_flat = q8_logits.view(-1, q8_logits.size(-1))
        q3_logits_flat = q3_logits.view(-1, q3_logits.size(-1))
        sst8_flat = sst8.view(-1)
        sst3_flat = sst3.view(-1)
        
        loss_q8 = self.criterion(q8_logits_flat, sst8_flat)
        loss_q3 = self.criterion(q3_logits_flat, sst3_flat)
        loss = loss_q8 + loss_q3
        
        q8_preds = torch.argmax(q8_logits, dim=-1)
        q3_preds = torch.argmax(q3_logits, dim=-1)
        mask = sst8 != -1
        
        self.train_q8_preds.extend(q8_preds[mask].cpu().numpy())
        self.train_q8_targets.extend(sst8[mask].cpu().numpy())
        self.train_q3_preds.extend(q3_preds[mask].cpu().numpy())
        self.train_q3_targets.extend(sst3[mask].cpu().numpy())
        
        self.log_dict({
            "train_loss": loss,
            "train_loss_q8": loss_q8,
            "train_loss_q3": loss_q3
        }, prog_bar=True)
        
        trackio.log({
            "train_loss": loss.item(),
            "train_loss_q8": loss_q8.item(),
            "train_loss_q3": loss_q3.item()
        })
        
        return loss
    
    def validation_step(self, batch, batch_idx):
        seqs, sst8, sst3, lengths = batch
        q8_logits, q3_logits = self(seqs, lengths)
        
        q8_logits_flat = q8_logits.view(-1, q8_logits.size(-1))
        q3_logits_flat = q3_logits.view(-1, q3_logits.size(-1))
        sst8_flat = sst8.view(-1)
        sst3_flat = sst3.view(-1)
        
        loss_q8 = self.criterion(q8_logits_flat, sst8_flat)
        loss_q3 = self.criterion(q3_logits_flat, sst3_flat)
        loss = loss_q8 + loss_q3
        
        q8_preds = torch.argmax(q8_logits, dim=-1)
        q3_preds = torch.argmax(q3_logits, dim=-1)
        mask = sst8 != -1
        
        self.val_q8_preds.extend(q8_preds[mask].cpu().numpy())
        self.val_q8_targets.extend(sst8[mask].cpu().numpy())
        self.val_q3_preds.extend(q3_preds[mask].cpu().numpy())
        self.val_q3_targets.extend(sst3[mask].cpu().numpy())
        
        self.log_dict({
            "val_loss": loss,
            "val_loss_q8": loss_q8,
            "val_loss_q3": loss_q3
        }, prog_bar=True)
        
        trackio.log({
            "val_loss": loss.item(),
            "val_loss_q8": loss_q8.item(),
            "val_loss_q3": loss_q3.item()
        })
        
        return loss
    
    def on_train_epoch_end(self):
        if len(self.train_q8_preds) > 0:
            f1_q8 = f1_score(self.train_q8_targets, self.train_q8_preds, average='macro')
            f1_q3 = f1_score(self.train_q3_targets, self.train_q3_preds, average='macro')
            harmonic_mean = 2 * (f1_q8 * f1_q3) / (f1_q8 + f1_q3 + 1e-8)
            
            self.log_dict({
                "train_f1_q8": f1_q8,
                "train_f1_q3": f1_q3,
                "train_harmonic_f1": harmonic_mean
            }, prog_bar=True)
            
            trackio.log({
                "train_f1_q8": f1_q8,
                "train_f1_q3": f1_q3,
                "train_harmonic_f1": harmonic_mean
            })
            
            self.train_q8_preds = []
            self.train_q8_targets = []
            self.train_q3_preds = []
            self.train_q3_targets = []
    
    def on_validation_epoch_end(self):
        if len(self.val_q8_preds) > 0:
            f1_q8 = f1_score(self.val_q8_targets, self.val_q8_preds, average='macro')
            f1_q3 = f1_score(self.val_q3_targets, self.val_q3_preds, average='macro')
            harmonic_mean = 2 * (f1_q8 * f1_q3) / (f1_q8 + f1_q3 + 1e-8)
            
            self.log_dict({
                "val_f1_q8": f1_q8,
                "val_f1_q3": f1_q3,
                "val_harmonic_f1": harmonic_mean
            }, prog_bar=True)
            
            trackio.log({
                "val_f1_q8": f1_q8,
                "val_f1_q3": f1_q3,
                "val_harmonic_f1": harmonic_mean
            })
            
            self.val_q8_preds = []
            self.val_q8_targets = []
            self.val_q3_preds = []
            self.val_q3_targets = []
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

print("✓ Lightning Module defined")

✓ Lightning Module defined


# Model Configuration


In [14]:
# Cell 13: Model Configuration
config = {
    'vocab_size': len(amino_acids),
    'embedding_dim': 128,
    'hidden_dim': 256,
    'num_q8_classes': len(q8_labels),
    'num_q3_classes': len(q3_labels),
    'num_layers': 2,
    'dropout': 0.3
}

print("Model Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

Model Configuration:
  vocab_size: 21
  embedding_dim: 128
  hidden_dim: 256
  num_q8_classes: 8
  num_q3_classes: 3
  num_layers: 2
  dropout: 0.3


# Initialize TrackIO


In [15]:
# Cell 14: Initialize TrackIO (Alternative - Without Auto Space Creation)
# Option 1: If you already created the space manually on HuggingFace
try:
    os.environ["HF_TOKEN"] = user_secrets.get_secret("dra_hf_access_token")
    
    trackio.init(
        project="25-t3-nppe2",
        space_id="Nv1023/dlgenai-nppe",
        name="bilstm-protein",
        group="bilstm"
    )
    print("✓ TrackIO initialized")
except Exception as e:
    print(f"TrackIO initialization failed: {e}")
    print("Continuing without TrackIO logging...")
    
    # Create a dummy trackio object to prevent errors
    class DummyTrackIO:
        def log(self, *args, **kwargs):
            pass
        def finish(self):
            pass
    
    import sys
    sys.modules['trackio'].log = lambda x: None
    sys.modules['trackio'].finish = lambda: None

* Trackio project initialized: 25-t3-nppe2
* Trackio metrics will be synced to Hugging Face Dataset: Nv1023/dlgenai-nppe-dataset
* Found existing space: https://huggingface.co/spaces/Nv1023/dlgenai-nppe
* View dashboard by going to: https://Nv1023-dlgenai-nppe.hf.space/


* Created new run: bilstm-protein
✓ TrackIO initialized


# Train BiRNN Model


In [16]:
# Cell 15: Train BiRNN Model
# COMMENT THIS ENTIRE CELL BEFORE SUBMISSION


base_model_rnn = BiRNN(
    vocab_size=config['vocab_size'],
    embedding_dim=config['embedding_dim'],
    hidden_dim=config['hidden_dim'],
    num_q8_classes=config['num_q8_classes'],
    num_q3_classes=config['num_q3_classes'],
    num_layers=config['num_layers'],
    dropout=config['dropout']
)

model_rnn = ProteinStructureModel(base_model_rnn)

checkpoint_callback_rnn = ModelCheckpoint(
    monitor='val_harmonic_f1',
    mode='max',
    dirpath='/kaggle/working/protein-birnn-checkpoints',
    filename='birnn-{epoch:02d}-{val_harmonic_f1:.4f}',
    save_top_k=1,
    verbose=True
)

early_stop_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    mode='min',
    verbose=True
)

trainer_rnn = Trainer(
    max_epochs=30,
    accelerator="auto",
    devices="auto",
    callbacks=[checkpoint_callback_rnn, early_stop_callback],
    logger=False,
)

print("=" * 60)
print("Training Bidirectional RNN Model")
print("=" * 60)

trainer_rnn.fit(model_rnn, train_loader, val_loader)

print(f"Best BiRNN model saved at: {checkpoint_callback_rnn.best_model_path}")


print("Training cell commented out")

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Training Bidirectional RNN Model


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | model     | BiRNN            | 600 K  | train
1 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
600 K     Trainable params
0         Non-trainable params
600 K     Total params
2.401     Total estimated model params size (MB)
7         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 18, global step 1957: 'val_harmonic_f1' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_loss did not improve in the last 5 records. Best score: 1.845. Signaling Trainer to stop.
Epoch 19, global step 2060: 'val_harmonic_f1' was not in top 1


Best BiRNN model saved at: /kaggle/working/protein-birnn-checkpoints/birnn-epoch=17-val_harmonic_f1=0.4292.ckpt
Training cell commented out


In [17]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


In [18]:
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers,
                 num_q8_classes, num_q3_classes, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
        )
        self.dropout = nn.Dropout(0.3)
        self.fc_q8 = nn.Linear(hidden_dim * 2, num_q8_classes)
        self.fc_q3 = nn.Linear(hidden_dim * 2, num_q3_classes)

    def forward(self, x, lengths):
        # x: (B, T)
        emb = self.embedding(x)  # (B, T, E)
        packed = pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True)  # (B, T, 2H)
        out = self.dropout(out)
        logits_q8 = self.fc_q8(out)  # (B, T, num_q8)
        logits_q3 = self.fc_q3(out)  # (B, T, num_q3)
        return logits_q8, logits_q3


In [19]:
class BiGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_q8_classes,
                 num_q3_classes, num_layers=1, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc_q8 = nn.Linear(hidden_dim * 2, num_q8_classes)
        self.fc_q3 = nn.Linear(hidden_dim * 2, num_q3_classes)

    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed = pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.gru(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True)
        out = self.dropout(out)
        logits_q8 = self.fc_q8(out)
        logits_q3 = self.fc_q3(out)
        return logits_q8, logits_q3


# Train BiGRU Model


In [20]:
# Cell 17: Train BiGRU Model (TRAINING NOTEBOOK ONLY)

base_model_gru = BiGRU(
    vocab_size=config['vocab_size'],
    embedding_dim=config['embedding_dim'],
    hidden_dim=config['hidden_dim'],
    num_q8_classes=config['num_q8_classes'],
    num_q3_classes=config['num_q3_classes'],
    num_layers=config['num_layers'],
    dropout=config['dropout']
)

model_gru = ProteinStructureModel(base_model_gru)

checkpoint_callback_gru = ModelCheckpoint(
    monitor='val_harmonic_f1',
    mode='max',
    dirpath='/kaggle/working/protein-bigru-checkpoints',
    filename='bigru-{epoch:02d}-{val_harmonic_f1:.4f}',
    save_top_k=1,
    verbose=True
)

early_stop_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    mode='min',
    verbose=True
)

trainer_gru = Trainer(
    max_epochs=30,
    accelerator="auto",
    devices="auto",
    callbacks=[checkpoint_callback_gru, early_stop_callback],
    logger=False,
)

print("=" * 60)
print("Training Bidirectional GRU Model")
print("=" * 60)

trainer_gru.fit(model_gru, train_loader, val_loader)

print(f"Best BiGRU model saved at: {checkpoint_callback_gru.best_model_path}")


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | model     | BiGRU            | 1.8 M  | train
1 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
1.8 M     Trainable params
0         Non-trainable params
1.8 M     Total params
7.136     Total estimated model params size (MB)
7         Modules in train mode
0         Modules in eval mode


Training Bidirectional GRU Model


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 10, global step 1133: 'val_harmonic_f1' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_loss did not improve in the last 5 records. Best score: 1.773. Signaling Trainer to stop.
Epoch 11, global step 1236: 'val_harmonic_f1' reached 0.44638 (best 0.44638), saving model to '/kaggle/working/protein-bigru-checkpoints/bigru-epoch=11-val_harmonic_f1=0.4464.ckpt' as top 1


Best BiGRU model saved at: /kaggle/working/protein-bigru-checkpoints/bigru-epoch=11-val_harmonic_f1=0.4464.ckpt


In [21]:
import os

print(os.listdir("/kaggle/working/protein-bigru-checkpoints"))


['bigru-epoch=11-val_harmonic_f1=0.4464.ckpt']


# Upload Models to KaggleHub


In [22]:
# Cell 18: Upload Models to KaggleHub
# COMMENT THIS ENTIRE CELL BEFORE SUBMISSION


KAGGLE_USERNAME = "nagavengadeshwaran"

# Upload BiRNN
handle_rnn = f"{KAGGLE_USERNAME}/protein-birnn/pytorch/v1"
kagglehub.model_upload(handle_rnn, checkpoint_callback_rnn.best_model_path, 
                       version_notes="BiRNN model for protein structure prediction")

# # Upload BiLSTM
# handle_lstm = f"{KAGGLE_USERNAME}/protein-bilstm/pytorch/v1"
# kagglehub.model_upload(handle_lstm, checkpoint_callback_lstm.best_model_path, 
#                        version_notes="BiLSTM model for protein structure prediction")


handle_gru = f"{KAGGLE_USERNAME}/protein-bigru/pytorch/v1"
kagglehub.model_upload(handle_gru, checkpoint_callback_gru.best_model_path, 
                       version_notes="BiGRU model for protein structure prediction")


trackio.finish()

print("Model upload cell commented out")

Uploading Model https://www.kaggle.com/models/nagavengadeshwaran/protein-birnn/pytorch/v1 ...
Starting upload for file /kaggle/working/protein-birnn-checkpoints/birnn-epoch=17-val_harmonic_f1=0.4292.ckpt


Uploading: 100%|██████████| 7.23M/7.23M [00:00<00:00, 15.0MB/s]

Upload successful: /kaggle/working/protein-birnn-checkpoints/birnn-epoch=17-val_harmonic_f1=0.4292.ckpt (7MB)


Your model instance version has been created.
Files are being processed...
See at: https://www.kaggle.com/models/nagavengadeshwaran/protein-birnn/pytorch/v1
Uploading Model https://www.kaggle.com/models/nagavengadeshwaran/protein-bigru/pytorch/v1 ...
Starting upload for file /kaggle/working/protein-bigru-checkpoints/bigru-epoch=11-val_harmonic_f1=0.4464.ckpt


Uploading: 100%|██████████| 21.4M/21.4M [00:00<00:00, 29.8MB/s]

Upload successful: /kaggle/working/protein-bigru-checkpoints/bigru-epoch=11-val_harmonic_f1=0.4464.ckpt (20MB)


Your model instance version has been created.
Files are being processed...
See at: https://www.kaggle.com/models/nagavengadeshwaran/protein-bigru/pytorch/v1
* Run finished. Uploading logs to Trackio (please wait...)
Model upload cell commented out


In [23]:
MODEL_TYPE = 'birnn'  # or 'bigru'

MODEL_FILES = {
    'birnn': '/kaggle/input/birnn-dataset-v1/birnn-epoch=17-val_harmonic_f1=0.4292.ckpt',
    'bigru': '/kaggle/input/bigru-dataset-v1/bigru-epoch=09-val_harmonic_f1=0.5012.ckpt',
}

checkpoint_path = MODEL_FILES[MODEL_TYPE]
print("Checkpoint:", checkpoint_path)


Checkpoint: /kaggle/input/birnn-dataset-v1/birnn-epoch=17-val_harmonic_f1=0.4292.ckpt


# Load Best BiRNN Model for Inference


In [24]:
# Cell 19: Load Best BiRNN Model for Inference (direct from working dir)

import torch
import os

checkpoint_path = "/kaggle/working/protein-birnn-checkpoints/birnn-epoch=17-val_harmonic_f1=0.4292.ckpt"
print("Using checkpoint:", checkpoint_path)
print("Exists:", os.path.exists(checkpoint_path))

base_model = BiRNN(
    vocab_size=config['vocab_size'],
    embedding_dim=config['embedding_dim'],
    hidden_dim=config['hidden_dim'],
    num_q8_classes=config['num_q8_classes'],
    num_q3_classes=config['num_q3_classes'],
    num_layers=config['num_layers'],
    dropout=config['dropout'],
)

model = ProteinStructureModel.load_from_checkpoint(checkpoint_path, model=base_model)
model.eval()
model.freeze()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("✓ BIRNN model loaded successfully!")
print("Device:", device)


Using checkpoint: /kaggle/working/protein-birnn-checkpoints/birnn-epoch=17-val_harmonic_f1=0.4292.ckpt
Exists: True
✓ BIRNN model loaded successfully!
Device: cuda


# Inference and submission.csv creation


In [25]:
# Recreate label index mappings for Q8 and Q3

Q8_COL = "sst8"
Q3_COL = "sst3"

q8_chars = sorted(list(set("".join(train_df[Q8_COL].tolist()))))
q8_label2idx = {c: i for i, c in enumerate(q8_chars)}
q8_idx2label = {i: c for c, i in q8_label2idx.items()}

q3_chars = sorted(list(set("".join(train_df[Q3_COL].tolist()))))
q3_label2idx = {c: i for i, c in enumerate(q3_chars)}
q3_idx2label = {i: c for c, i in q3_label2idx.items()}

print("Q8 mapping:", q8_label2idx)
print("Q3 mapping:", q3_label2idx)


Q8 mapping: {'B': 0, 'C': 1, 'E': 2, 'G': 3, 'H': 4, 'I': 5, 'S': 6, 'T': 7}
Q3 mapping: {'C': 0, 'E': 1, 'H': 2}


In [26]:
# Cell B: Inference and submission.csv creation

model.eval()
device = next(model.parameters()).device

all_ids = []
all_pred_q8 = []
all_pred_q3 = []

with torch.no_grad():
    for batch in test_loader:
        x, ids, lengths = batch  # x: Tensor, ids: tuple, lengths: Tensor

        x = x.to(device)
        lengths = lengths.to(device)

        # Convert ids tuple -> list for indexing
        ids = list(ids)

        # Forward through Lightning model (must accept x, lengths)
        logits_q8, logits_q3 = model(x, lengths)

        preds_q8 = logits_q8.argmax(-1).cpu().numpy()
        preds_q3 = logits_q3.argmax(-1).cpu().numpy()
        lengths_np = lengths.cpu().numpy()

        for i, L in enumerate(lengths_np):
            seq_q8_idx = preds_q8[i, :L]
            seq_q3_idx = preds_q3[i, :L]

            seq_q8 = "".join(q8_idx2label[int(t)] for t in seq_q8_idx)
            seq_q3 = "".join(q3_idx2label[int(t)] for t in seq_q3_idx)

            all_ids.append(int(ids[i]))
            all_pred_q8.append(seq_q8)
            all_pred_q3.append(seq_q3)

import pandas as pd

submission = pd.DataFrame({
    "id": all_ids,
    "sst8": all_pred_q8,
    "sst3": all_pred_q3,
})

submission = submission.sort_values("id").reset_index(drop=True)
submission.to_csv("submission.csv", index=False)
submission.head()


,id,sst8,sst3
0,0,TTTTTTBBBBBBBBBBBBBBBBBTTGGGGGGTTTTITTTGGGGGTI...,HHHHHHCCCCCCCCCCCCCCCCCHHEEEEEEHHHHHHHHHEEEEHH...
1,1,TTTTTTTTTTTTGGGGGTSIIGGGGGGGTIIBBBBBBBBTTTIIBB...,HHHHHHHHHHHHEEEEEHHHHEEEEEEEHHHCCCCCCCCHHHHHCC...
2,2,TTGGGGGGBTIBBBBBBBBBBBBBBBBBBBBBBBBTTIBBBBBBBB...,HHHCECCCHHHCCCCCCCCCCCCCCCCCCCCCCCHHHHCCCCCCCC...
3,3,TTTIBBBBBBBBBBBBBBBBBBBBBBBBTTGGGGIIIBBBBBBBBB...,HHHHHCCCCCCCCCCCCCCCCCCCCCCCHHEEEEHHCCCCCCCCCC...
4,4,TTGGGGGGTTTIIBBBBBBBBBBBBBBTTTTTTTIIITGGGGTSGG...,HHEEEEEEHHHHHCCCCCCCCCCCCCCHHHHHHHHHHHEEEEHHHE...
